In [1]:
import os
import re
import gc
import sqlite3
import unicodedata
import time
from pathlib import Path

import pandas as pd
import numpy as np

print("Environment ready")

BASE = "/kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource"

TRAIN = f"{BASE}/dataset/train"
TEST  = f"{BASE}/dataset/test"

WORK = "/kaggle/working/amazon_ml"
os.makedirs(WORK, exist_ok=True)

paths = {
    "train_s1": f"{TRAIN}/train_source1.tsv",
    "train_s2": f"{TRAIN}/train_source2.tsv",
    "train_s3": f"{TRAIN}/train_source3.tsv",
    "ground_truth": f"{TRAIN}/train_ground_truth.tsv",

    "test_s1": f"{TEST}/test_source1.tsv",
    "test_s2": f"{TEST}/test_source2.tsv",
    "test_s3": f"{TEST}/test_source3.tsv",
}

for k, v in paths.items():
    print(k, "->", v, os.path.exists(v))

Environment ready
train_s1 -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv True
train_s2 -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv True
train_s3 -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv True
ground_truth -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv True
test_s1 -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/test/test_source1.tsv True
test_s2 -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/test/test_source2.tsv True
test_s3 -> /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_reso

In [2]:
!nvidia-smi

Sat Sep 26 08:36:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
LEGAL_SUFFIXES = {
    "inc",
    "incorporated",
    "corp",
    "corporation",
    "ltd",
    "limited",
    "llc",
    "co",
    "company",
    "pvt",
    "pvt ltd",
    "private limited",
}


ADDRESS_REPLACEMENTS = {
    "street": "st",
    "road": "rd",
    "avenue": "ave",
    "boulevard": "blvd",
    "drive": "dr",
    "lane": "ln",
    "highway": "hwy",
    "apartment": "apt",
    "floor": "fl",
    "suite": "ste",
}


def normalize_unicode(text):
    if text is None:
        return ""

    text = str(text).strip()

    if not text:
        return ""

    # Unicode compatibility normalization
    text = unicodedata.normalize("NFKC", text).casefold()

    out = []

    for ch in text:
        category = unicodedata.category(ch)

        # Keep letters, numbers and combining marks.
        if (
            category.startswith("L")
            or category.startswith("N")
            or category.startswith("M")
        ):
            out.append(ch)

        elif ch.isspace():
            out.append(" ")

        else:
            out.append(" ")

    return re.sub(r"\s+", " ", "".join(out)).strip()


def normalize_name(text):
    text = normalize_unicode(text)

    if not text:
        return ""

    # & -> and
    text = text.replace("&", " and ")

    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    # Remove legal suffixes only at the END
    while words:
        if words[-1] in LEGAL_SUFFIXES:
            words.pop()

        elif (
            len(words) >= 2
            and " ".join(words[-2:]) in LEGAL_SUFFIXES
        ):
            words = words[:-2]

        else:
            break

    return " ".join(words)


def normalize_address(text):
    text = normalize_unicode(text)

    if not text:
        return ""

    words = text.split()

    return " ".join(
        ADDRESS_REPLACEMENTS.get(word, word)
        for word in words
    )


print(normalize_name("राम मार्केटिंग प्राइवेट लिमिटेड"))
print(normalize_name("मॉडर्न फाइनेंस"))
print(normalize_name("Payne Énterprises LLC"))
print(normalize_address("3315 Fremont Street, Peoria, IL"))

राम मार्केटिंग प्राइवेट लिमिटेड
मॉडर्न फाइनेंस
payne énterprises
3315 fremont st peoria il


In [4]:
gt = pd.read_csv(
    paths["ground_truth"],
    sep="\t",
    dtype=str,
    keep_default_na=False
)

gt["true_ids"] = gt["matched_entity_ids"].apply(
    lambda x: set(
        i.strip()
        for i in x.split(",")
        if i.strip()
    )
)

print("Ground truth rows:", len(gt))
print("Singleton/no-match S1:", (gt["matched_entity_ids"] == "").sum())

gt.head()

Ground truth rows: 2206821
Singleton/no-match S1: 123247


,source1_entity_id,matched_entity_ids,true_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129...","{S2-681193310, S3-860443364, S2-743505751, S3-..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843...","{S2-249013014, S3-384364074, S2-197070651, S3-..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467","{S3-878454467, S2-790675320, S2-479876582}"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215","{S2-24659151, S3-679606215, S2-153058913}"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280...","{S3-449308785, S2-625774905, S2-478959098, S3-..."


In [5]:
SAMPLE_N = 1000

s1_sample = pd.read_csv(
    paths["train_s1"],
    sep="\t",
    dtype=str,
    nrows=SAMPLE_N,
    keep_default_na=False
)

print("S1 validation sample:", len(s1_sample))
print(s1_sample.head())

S1 validation sample: 1000
      entity_id            business_name  \
0  S1-925783039      Orelee's Barbershop   
1  S1-773889195              Prime Money   
2  S1-377745466            B+ Retail Inc   
3  S1-133037285            Christ Chapel   
4  S1-755362802  Prabhav Business Center   

                                    business_address country  
0             1795 Westchester Drive, High Point, NC      US  
1                    17560 Ellis Road, Tahlequah, OK      US  
2                1712 Montebello Avenue, Phoenix, AZ      US  
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US  
4  797, Lake Town Block A, Kolkata, Howrah, West ...   India  


In [6]:
# ============================================================
# CELL 6 — Build SQLite index safely
# ============================================================

import sqlite3
import os
import time
import gc

DB_PATH = f"{WORK}/entity_index.sqlite"

# Remove partially-created database
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

conn.execute("""
CREATE TABLE entities (
    entity_id TEXT PRIMARY KEY,
    source TEXT NOT NULL,
    business_name TEXT,
    business_address TEXT,
    country TEXT,
    name_norm TEXT,
    address_norm TEXT
)
""")

conn.execute("""
CREATE INDEX idx_name_country
ON entities(source, country, name_norm)
""")

conn.execute("""
CREATE INDEX idx_address_country
ON entities(source, country, address_norm)
""")

conn.commit()


def build_sqlite_index(path, source, chunksize=100_000):

    start = time.time()
    total = 0

    for chunk in pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        chunksize=chunksize
    ):

        # Normalize
        chunk["name_norm"] = chunk["business_name"].map(normalize_name)
        chunk["address_norm"] = chunk["business_address"].map(normalize_address)

        rows = chunk[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
                "name_norm",
                "address_norm"
            ]
        ].copy()

        rows["source"] = source

        rows = rows[
            [
                "entity_id",
                "source",
                "business_name",
                "business_address",
                "country",
                "name_norm",
                "address_norm"
            ]
        ]

        # IMPORTANT:
        # method="multi" caused "too many SQL variables".
        # method=None lets pandas/SQLite insert safely.
        rows.to_sql(
            "entities",
            conn,
            if_exists="append",
            index=False,
            method=None,
            chunksize=1000
        )

        total += len(rows)

        print(
            f"\r{source}: {total:,} rows",
            end=""
        )

        del chunk, rows
        gc.collect()

    print(
        f"\nFinished {source}: "
        f"{total:,} rows in {time.time() - start:.1f}s"
    )


build_sqlite_index(
    paths["train_s2"],
    "S2"
)

build_sqlite_index(
    paths["train_s3"],
    "S3"
)

conn.commit()

print("\nSQLite index ready.")

S2: 5,034,616 rows
Finished S2: 5,034,616 rows in 705.5s
S3: 5,285,603 rows
Finished S3: 5,285,603 rows in 753.4s

SQLite index ready.
